# 🎯 Huấn luyện mô hình Random Forest (Final Optimized)
Dựa trên phân tích thực tế, mô hình Decision Tree bị Overfitting. Tệp này chứa quy trình chuẩn hóa dữ liệu bằng `RobustScaler` và huấn luyện mô hình `Random Forest Classifier` để đạt hiệu suất tối ưu.

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

## 1. Load và Tiền xử lý dữ liệu

In [2]:
print("1️⃣ Loading dataset...")
columns_to_keep = [
    'Seq', 'Mean', 'sTos', 'sTtl', 'dTtl', 'sHops', 'TotBytes', 
    'SrcBytes', 'Offset', 'sMeanPktSz', 'dMeanPktSz', 'SrcWin', 
    'TcpRtt', 'AckDat', 'Label', ' e        ', ' e d      ', 
    'icmp', 'tcp', 'CON', 'FIN', 'INT', 'REQ', 'RST', 'Status'
]

df = pd.read_csv(r'D:\Study\DH\IoT in 5G\dataset\dataset5g-nidd\Encoded\Encoded.csv', usecols=columns_to_keep)

# Xử lý missing values
missing_counts = df.isnull().sum()
if missing_counts.sum() > 0:
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if df[col].isnull().any():
            df[col].fillna(df[col].median(), inplace=True)
    if df['Label'].isnull().any():
        df.dropna(subset=['Label'], inplace=True)

# Xử lý infinite values
for col in df.select_dtypes(include=[np.number]).columns:
    if np.isinf(df[col]).any():
        df[col].replace([np.inf, -np.inf], df[col].median(), inplace=True)

print(f"✓ Loaded {len(df):,} samples")

1️⃣ Loading dataset...


✓ Loaded 1,215,890 samples


## 2. (Skipped) Feature Engineering
Sử dụng đúng 24 features gốc như bài báo, không thêm feature phái sinh.

In [3]:
# Move Label to last temporarily
column_to_move = df.pop('Label')

# Put Label back
df['Label'] = column_to_move

feature_names = df.columns[:-1].tolist()
print(f"✓ Using {len(feature_names)} features")

✓ Using 24 features


## 3. Scale Dữ liệu và Tách tập Train/Test

In [4]:
X = df.iloc[:, :-1].values
y = df['Label'].values

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"✓ Data prepared: Train={len(X_train):,}, Val={len(X_val):,}, Test={len(X_test):,}")

✓ Data prepared: Train=729,534, Val=243,178, Test=243,178


## 4. Huấn luyện Random Forest

In [5]:
print("Training Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=100,
    min_samples_leaf=50,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_scaled, y_train)
print("✓ Training complete!")

Training Random Forest...


✓ Training complete!


## 5. Đánh giá Mô hình trên tập Test

In [6]:
y_pred = rf_model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9996

Classification Report:


              precision    recall  f1-score   support

      Benign       1.00      1.00      1.00     95547
   Malicious       1.00      1.00      1.00    147631

    accuracy                           1.00    243178
   macro avg       1.00      1.00      1.00    243178
weighted avg       1.00      1.00      1.00    243178

Confusion Matrix:


[[ 95517     30]
 [    59 147572]]


## 6. Lưu Mô hình

In [7]:
from datetime import datetime
import os
os.makedirs('../model', exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_path = f'../model/random_forest_model_OPTIMIZED_{timestamp}.pkl'
scaler_path = f'../model/scaler_OPTIMIZED_{timestamp}.pkl'
features_path = f'../model/feature_names_OPTIMIZED_{timestamp}.pkl'

joblib.dump(rf_model, model_path)
joblib.dump(scaler, scaler_path)
joblib.dump(feature_names, features_path)

print(f"💾 Best model saved:")
print(f"  Model:    {model_path}")
print(f"  Scaler:   {scaler_path}")
print(f"  Features: {features_path}")

💾 Best model saved:
  Model:    ../model/random_forest_model_OPTIMIZED_20260603_222933.pkl
  Scaler:   ../model/scaler_OPTIMIZED_20260603_222933.pkl
  Features: ../model/feature_names_OPTIMIZED_20260603_222933.pkl
